# Cortex AI Gateway + LangChain Agent + MCP

This notebook demonstrates how to build a **LangChain agent** that:

1. Routes inference through **Snowflake's Cortex AI Gateway** (using a Snowflake-hosted model)
2. Connects to **Snowflake data via MCP** (Model Context Protocol) for tool use
3. Produces **observability traces** you can query in `SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE`

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     LangChain Agent (ReAct)                     │
│                                                                 │
│   LLM: ChatOpenAI ──► Cortex AI Gateway ──► LLM (Cortex)       │
│                                                                 │
│   Tools: MCP Client ──► Snowflake MCP Server                   │
│          ├── Cortex Analyst (SQL via semantic view)             │
│          └── Cortex Search (RAG over strategy docs)            │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
            SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE
            SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY
```

## Prerequisites

1. Run `setup.sql` in your Snowflake account to create the database, tables, semantic view, search service, MCP server, and gateway route.
2. Create a [Personal Access Token (PAT)](https://docs.snowflake.com/en/user-guide/admin-pat) in Snowflake for API authentication.
3. Have your Snowflake account identifier (e.g., `myorg-myaccount`).

## 1. Install Dependencies

In [1]:
%pip install "langchain-openai>=1.6" "langchain-mcp-adapters>=0.3" "langgraph>=1.0,<2" "snowflake-connector-python>=3.12"


Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

We read credentials from `~/.snowflake/connections.toml` (the standard Snowflake CLI config).
Set `SNOWFLAKE_CONNECTION_NAME` to match the connection you want to use.

The AI Gateway has two distinct URL paths:
- **Inference:** `/api/v2/aigateways/snowflake/v1/chat/completions` (OpenAI-compatible) or `/v1/messages` (Anthropic)
- **Admin:** `/api/v2/aigateways/SNOWFLAKE` (spec management, SHOW, ALTER)

In [2]:
import os
import tomllib
from pathlib import Path

CONNECTION_NAME = os.environ.get("SNOWFLAKE_CONNECTION_NAME")

# Read connection config from connections.toml
connections_path = Path.home() / ".snowflake" / "connections.toml"
if not connections_path.exists():
    # Fall back to Cortex agent config location
    connections_path = Path.home() / ".snowflake" / "cortex" / "agent" / "connections.toml"

with open(connections_path, "rb") as f:
    connections = tomllib.load(f)

if CONNECTION_NAME is None:
    if len(connections) == 1:
        CONNECTION_NAME = next(iter(connections))
    else:
        raise SystemExit("Set SNOWFLAKE_CONNECTION_NAME to one of: " + ", ".join(connections))

conn_cfg = connections[CONNECTION_NAME]
SNOWFLAKE_ACCOUNT = conn_cfg["account"]
SNOWFLAKE_USER = conn_cfg["user"]

# The PAT may be stored under either key depending on how the connection was
# created. An SSO/externalbrowser connection has neither -- the gateway and MCP
# endpoints are REST APIs that require a token, so a PAT is mandatory here.
SNOWFLAKE_PAT = conn_cfg.get("password") or conn_cfg.get("token") or os.environ.get("SNOWFLAKE_PAT")
if not SNOWFLAKE_PAT:
    raise SystemExit(
        f"No PAT found for connection '{CONNECTION_NAME}'. Add a PAT as the 'password' "
        "key in connections.toml, or export SNOWFLAKE_PAT. Create one with:\n"
        "  ALTER USER <you> ADD PROGRAMMATIC ACCESS TOKEN <name>\n"
        "      ROLE_RESTRICTION = AI_GATEWAY_USER DAYS_TO_EXPIRY = 30;\n"
        "If you use ROLE_RESTRICTION, run section 8 of setup.sql to grant that role "
        "access to the lab objects."
    )

# Snowflake hostnames must use hyphens, not underscores, for valid SSL certs.
# The base_url reported by SHOW AI GATEWAYS echoes the account name verbatim and
# may contain underscores, so it needs this same transform before use.
SF_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace("_", "-")

# Two distinct gateway URLs:
#   Inference -> /api/v2/aigateways/snowflake/v1
#   Admin     -> /api/v2/aigateways/SNOWFLAKE
#
# Do NOT point inference at /api/v2/cortex/v1. That is the separate Cortex
# Inference REST API. It answers successfully, so a wrong base_url fails
# silently -- but those calls bypass the gateway entirely: they never reach
# AI_GATEWAY_USAGE_HISTORY or gateway traces, and ignore the model allowlist,
# budgets, and quotas.
INFERENCE_BASE_URL = f"https://{SF_HOST}/api/v2/aigateways/snowflake/v1"
GATEWAY_ADMIN_URL = f"https://{SF_HOST}/api/v2/aigateways/SNOWFLAKE"

# The model name is passed through to the gateway, whose 'models' allowlist
# decides what is permitted (our spec uses '*'). Availability still varies by
# region -- section 3 doubles as an availability check.
MODEL = os.environ.get("GATEWAY_MODEL", "openai-gpt-5.4")

print(f"Connection:    {CONNECTION_NAME}")
print(f"Account:       {SNOWFLAKE_ACCOUNT}")
print(f"User:          {SNOWFLAKE_USER}")
print(f"Inference URL: {INFERENCE_BASE_URL}")
print(f"Admin URL:     {GATEWAY_ADMIN_URL}")
print(f"Model:         {MODEL}")


Connection:    parker_demo
Account:       SFSENORTHAMERICA-PERICKSON_AWS1
User:          PERICKSON
Inference URL: https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/aigateways/snowflake/v1
Admin URL:     https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/aigateways/SNOWFLAKE
Model:         openai-gpt-5.4


## 3. Initialize the Gateway LLM

The Cortex AI Gateway exposes an **OpenAI-compatible** API. We use `ChatOpenAI` from LangChain,
pointing `base_url` at the inference endpoint and passing the PAT as the API key.

The `model` parameter is the actual model name (e.g., `openai-gpt-5.4`). The gateway's
`models` allowlist controls which models are permitted — our spec uses `'*'` to allow all.
Availability varies by region *and over time* even with cross-region inference enabled, so
the test call below doubles as an availability check. A `503` means the gateway could not
serve that model at that moment — often transient capacity rather than a misconfiguration,
so retry before concluding a model is unavailable to you.
Model availability varies by region even with cross-region inference enabled, so the cell
below doubles as an availability check: a `503` means the gateway cannot serve that model
in your account.

### Trace context propagation

For the gateway to group all inference calls from one agent turn under a single
`trace_id`, the client must send a [W3C `traceparent`](https://www.w3.org/TR/trace-context/) header on every request. Without it, each call gets its own trace ID and
they appear as unrelated events in `AGENT_TRACE_TABLE`.

We define a `make_llm()` factory that accepts an optional `traceparent` header.
In `run_agent()` (Section 5) we generate a fresh traceparent per invocation so
all LLM calls within one turn share the same trace.

In [3]:
import uuid
from langchain_openai import ChatOpenAI


def make_traceparent() -> tuple[str, str]:
    """Generate a W3C traceparent header value and return (header_value, trace_id).

    Format: {version}-{trace_id}-{parent_id}-{flags}
    See https://www.w3.org/TR/trace-context/#traceparent-header
    """
    trace_id = uuid.uuid4().hex  # 32 hex chars
    parent_id = uuid.uuid4().hex[:16]  # 16 hex chars
    header = f"00-{trace_id}-{parent_id}-01"
    return header, trace_id


def make_llm(traceparent: str | None = None) -> ChatOpenAI:
    """Create a ChatOpenAI pointed at the Cortex AI Gateway.

    When *traceparent* is provided it is sent as a default header on every
    request so the gateway groups all inference calls under one trace_id.
    """
    headers = {}
    if traceparent:
        headers["traceparent"] = traceparent
    return ChatOpenAI(
        model=MODEL,
        base_url=INFERENCE_BASE_URL,
        api_key=SNOWFLAKE_PAT,
        temperature=0,
        max_tokens=4096,
        default_headers=headers or None,
    )


# Quick test — this goes through the gateway to Claude
llm = make_llm()
response = llm.invoke("What is ROAS in marketing? Answer in one sentence.")
print(response.content)

ROAS, or Return on Ad Spend, is a marketing metric that measures how much revenue is generated for every dollar spent on advertising.


## 4. Connect MCP Tools

We use `langchain-mcp-adapters` to connect to the **Snowflake MCP server** object we created
in `setup.sql` (`MARKETING_MCP`). This MCP server exposes three tools:

- **`query_campaigns`** (Cortex Analyst) — translates natural language to SQL against the semantic view
- **`search_strategy_docs`** (Cortex Search) — retrieves relevant strategy documents via RAG
- **`execute_sql`** — runs the SQL that `query_campaigns` returns and gives back the rows

Cortex Analyst returns SQL, not data, so `execute_sql` is what makes the quantitative path
work end to end. The agent has to chain the two.

The MCP server endpoint follows the pattern:
```
https://<account>.snowflakecomputing.com/api/v2/databases/<db>/schemas/<schema>/mcp-servers/<name>
```

`langchain-mcp-adapters` speaks streamable HTTP directly — no local bridge or npx needed,
and no `/sse` suffix.

**If you get `MCP server ... does not exist or not authorized`:** your PAT is role-restricted
and that role lacks grants on the lab objects. Run section 8 of `setup.sql`.


In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP server endpoint — points to our MARKETING_MCP server object in Snowflake
# Snowflake hostnames require hyphens (not underscores) for valid SSL certs
MCP_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace('_', '-')
MCP_ENDPOINT = (
    f"https://{MCP_HOST}"
    f"/api/v2/databases/CORTEX_GATEWAY_LAB/schemas/PUBLIC/mcp-servers/MARKETING_MCP"
)

# Streamable HTTP transport — connects directly to the Snowflake MCP server
mcp_config = {
    "snowflake": {
        "transport": "http",
        "url": MCP_ENDPOINT,
        "headers": {"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    }
}

print("MCP server configured (will connect when agent starts)")
print(f"  Transport: http")
print(f"  Endpoint:  {MCP_ENDPOINT}")

MCP server configured (will connect when agent starts)
  Transport: http
  Endpoint:  https://SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com/api/v2/databases/CORTEX_GATEWAY_LAB/schemas/PUBLIC/mcp-servers/MARKETING_MCP


## 5. Build the LangChain Agent

We create a **ReAct agent** using LangGraph. The agent:
1. Receives a user question
2. Decides which MCP tool(s) to call (Cortex Analyst for data, Cortex Search for docs)
3. Routes LLM inference through the Cortex AI Gateway
4. Synthesizes a final answer from the tool results
> `create_react_agent` emits a deprecation warning on LangGraph 1.x — it moves to
> `langchain.agents.create_agent` in 2.0. It still works, and switching means adding the
> `langchain` package as a dependency, so the pin above stays on `langgraph<2`.


In [5]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a marketing analytics assistant with access to campaign performance
data and strategy documents via Snowflake. Use the available tools to answer questions:

- For quantitative questions about campaign spend, revenue, ROI, etc.:
  1. First call query_campaigns to generate the SQL query.
  2. Extract the SQL statement from the response.
  3. Then call execute_sql with that SQL to get the actual data rows.
- For questions about strategy, methodology, or planning, use search_strategy_docs.
- For questions that need both data and context, use both.

Always cite specific numbers when available. Express ROI as a multiplier (e.g., 3.2x).
Round currency to 2 decimal places."""

# Initialize the MCP client and load tools once
mcp_client = MultiServerMCPClient(mcp_config)
tools = await mcp_client.get_tools()
print(f"Loaded {len(tools)} MCP tools: {[t.name for t in tools]}")


async def run_agent(question: str):
    """Run the agent against a question.

    Each invocation creates a fresh LLM with a unique W3C traceparent header
    so that every inference call the agent makes during this turn is grouped
    under one trace_id in AGENT_TRACE_TABLE.
    """
    traceparent, trace_id = make_traceparent()
    traced_llm = make_llm(traceparent=traceparent)
    print(f"Trace ID: {trace_id}")

    agent = create_react_agent(
        model=traced_llm,
        tools=tools,
        prompt=SYSTEM_PROMPT,
    )

    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    # Print tool calls and responses for visibility
    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool call: {tc['name']}({tc['args']})")
        if msg.type == 'tool':
            print(f"Tool response ({msg.name}): {msg.content[:500]}")
            print()

    final_message = result["messages"][-1]
    print("─" * 60)
    print(f"Q: {question}")
    print("─" * 60)
    print(final_message.content)
    return result

Loaded 3 MCP tools: ['query_campaigns', 'search_strategy_docs', 'execute_sql']


## 6. Run Queries

Let's ask the agent some marketing analytics questions. Each request flows through:

**User → LangChain Agent → AI Gateway (LLM) → MCP Tools (Snowflake) → Response**

All inference calls are logged by the gateway for observability.

In [6]:
# Query 1: Structured data question (routes to Cortex Analyst via MCP)
result1 = await run_agent("What was the ROAS by channel for Q4 2024?")

Trace ID: b47153884f4c45198cb75111ac92ca21


/var/folders/nx/tk8s9m3n36ncjzn4lvvk2dfw0000gn/T/ipykernel_2440/1010971107.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: query_campaigns({'message': 'What was the ROAS by channel for Q4 2024? Return channel, total spend, total revenue, and ROAS for campaigns in Q4 2024.'})
Tool response (query_campaigns): [{'type': 'text', 'text': '[{"text":"This is our interpretation of your question:\\n\\nWhat was the ROAS by channel for Q4 2024 (October, November, December 2024)? Return channel, total spend (rounded to 2 decimal places), total revenue (rounded to 2 decimal places), and ROAS (rounded to 2 decimal places) for campaigns in Q4 2024."},{"statement":"SELECT *\\nFROM SEMANTIC_VIEW(\\n    CORTEX_GATEWAY_LAB.PUBLIC.CMO_ANALYTICS\\n    DIMENSIONS channel\\n    METRICS total_spend, total_revenue, roas\\n    WHERE month >= \'2024-10-01\' AND month < \'2025-01-01\'\\n)\\n -- Generated by Cortex Analyst (request_id: 50418d4e-baca-4812-8e7b-76b435734209)\\n;","confidence":{}}]', 'id': 'lc_2b0cc852-19d8-4b8a-87c4-235e42ac9a03'}]

Tool call: execute_sql({'sql': "SELECT *\nFROM SEMANTIC_VIEW(\n    CORTEX_GAT

In [7]:
# Query 2: Unstructured data question (routes to Cortex Search via MCP)
result2 = await run_agent("What is our attribution methodology and how is revenue assigned to channels?")

Trace ID: 5ab484aef78041d6996773c04a5c486a


/var/folders/nx/tk8s9m3n36ncjzn4lvvk2dfw0000gn/T/ipykernel_2440/1010971107.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: search_strategy_docs({'query': 'attribution methodology revenue assigned to channels marketing attribution model channel revenue assignment methodology', 'columns': ['CONTENT', 'TITLE', 'ATTRIBUTES'], 'limit': 5})
Tool response (search_strategy_docs): [{'type': 'text', 'text': 'MCP error calling tool search_strategy_docs: Error occurred while calling Cortex Search Service: Column ATTRIBUTES was not indexed in this Cortex Search Service\nrequest-id: 68956c5a-d898-43a1-9ff2-bb3a497efa76', 'id': 'lc_79a1de94-14cc-41ec-bdea-13f07fb055ec'}]

Tool call: search_strategy_docs({'query': 'attribution methodology revenue assigned to channels marketing attribution model channel revenue assignment methodology', 'columns': ['CONTENT', 'TITLE'], 'limit': 5})
Tool response (search_strategy_docs): [{'type': 'text', 'text': '[{"@scores":{"text_match":0.044712532,"cosine_similarity":0.56343573,"reranker_score":2.0857196},"TITLE":"Attribution Methodology","CONTENT":"We use a data-driven multi-t

In [8]:
# Query 3: Hybrid question (may use both Analyst + Search)
result3 = await run_agent(
    "How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? "
    "Include actual ROAS and conversion numbers."
)

Trace ID: edcd63cb98f5457c8832e532373480d7


/var/folders/nx/tk8s9m3n36ncjzn4lvvk2dfw0000gn/T/ipykernel_2440/1010971107.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Tool call: query_campaigns({'message': 'How did our Q4 Holiday Push campaign perform? Return spend, revenue, ROAS, conversions for the Q4 Holiday Push campaign, and if possible target metrics if stored in campaign data.'})
Tool call: search_strategy_docs({'query': 'Q4 planning brief Holiday Push targets ROAS conversions campaign goals', 'columns': ['CONTENT', 'TITLE', 'ATTRIBUTES'], 'limit': 5})
Tool response (query_campaigns): [{'type': 'text', 'text': '[{"text":"This is our interpretation of your question:\\n\\nHow did the \'Q4 Holiday Push\' campaign perform in 2024? Return total spend, total revenue, ROAS, and total conversions for the Q4 Holiday Push campaign, broken down by channel. Currency values rounded to 2 decimal places. No target metrics are stored in the campaign data schema, so only available metrics are returned. Date range defaults to full year 2024."},{"statement":"WITH __campaign_spend AS (\\n  SELECT\\n    campaign_name,\\n    channel,\\n    month,\\n    conversions

## 7. Observability — AI Gateway Traces

Every request through the AI Gateway is recorded with OpenTelemetry traces. These land in
the `AGENT_TRACE_TABLE('SNOWFLAKE')` table function, where each row is a **span** in a trace.

Key columns:
- `TRACE_ID` — groups all spans from one end-to-end request
- `SPAN_ID` — unique identifier for each operation
- `RECORD_ATTRIBUTES` — JSON with model name, token counts, etc.
- `TIMESTAMP` — when the span was recorded

Because we enabled `capture_payload.request_response: true` in the gateway spec,
request/response content is also captured in the trace attributes.

Let's query the observability data for the requests we just made.

In [9]:
import json

import snowflake.connector

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    authenticator="PROGRAMMATIC_ACCESS_TOKEN",
    token=SNOWFLAKE_PAT,
    warehouse="GATEWAY_LAB_WH",
)
cur = conn.cursor()
print(f"Connected as {SNOWFLAKE_USER} via PAT.")


Connected as PERICKSON via PAT.


In [10]:
# Recent AI Gateway traces
cur.execute("""
    SELECT
        TRACE:trace_id::STRING AS TRACE_ID,
        RECORD:name::STRING AS SPAN_NAME,
        RECORD_ATTRIBUTES:"gen_ai.request.model"::STRING AS MODEL,
        RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
        RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK,
        RECORD_ATTRIBUTES:"http.status_code"::INT AS HTTP
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 15
""")

rows = cur.fetchall()
print(f"Found {len(rows)} recent gateway trace spans\n")
print(f"{'TRACE_ID':>36} | {'SPAN_NAME':>25} | {'MODEL':>16} | {'IN':>6} | {'OUT':>6} | {'HTTP':>4}")
print("─" * 100)
for r in rows:
    print(f"{str(r[0]):>36} | {str(r[1]):>25} | {str(r[2]):>16} | {r[3]:>6} | {r[4]:>6} | {r[5]:>4}")

Found 15 recent gateway trace spans

                            TRACE_ID |                 SPAN_NAME |            MODEL |     IN |    OUT | HTTP
────────────────────────────────────────────────────────────────────────────────────────────────────
    edcd63cb98f5457c8832e532373480d7 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1945 |     40 |  200
    edcd63cb98f5457c8832e532373480d7 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1331 |    245 |  200
    edcd63cb98f5457c8832e532373480d7 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    783 |    111 |  200
    5ab484aef78041d6996773c04a5c486a |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1844 |    227 |  200
    5ab484aef78041d6996773c04a5c486a |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    881 |     42 |  200
    5ab484aef78041d6996773c04a5c486a |       chat openai-gpt-5.4 |   openai-gpt-5.4 |    769 |     45 |  200
    b47153884f4c45198cb75111ac92ca21 |       chat openai-gpt-5.4 |   openai-gpt-5.4 |   1441 |    2

In [11]:
# Credit usage summary from AI_GATEWAY_USAGE_HISTORY
# OPERATION_DETAILS is a JSON object keyed by model name with token counts
cur.execute("""
    SELECT
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY,
        LATERAL FLATTEN(input => OPERATION_DETAILS) f
    WHERE START_TIME > DATEADD('day', -1, CURRENT_TIMESTAMP())
    GROUP BY f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Usage Summary (last 24h)\n")
print(f"{'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 70)
for row in rows:
    print(f"{str(row[0]):>20} | {row[1]:>8} | {row[2]:>10} | {row[3]:>10} | {row[4]:>10.6f}")

AI Gateway Usage Summary (last 24h)

               MODEL | REQUESTS |  IN_TOKENS | OUT_TOKENS |    CREDITS
──────────────────────────────────────────────────────────────────────
      openai-gpt-5.4 |       11 |      10644 |       1345 |   0.023680


In [12]:
# Detailed trace view — follow a single request end-to-end
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
latest_trace = cur.fetchone()

if latest_trace:
    trace_id = latest_trace[0]
    print(f"Trace detail for: {trace_id}\n")

    cur.execute("""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD:name::STRING AS SPAN_NAME,
            TIMESTAMP,
            RECORD_ATTRIBUTES
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = %s
        ORDER BY TIMESTAMP
    """, (trace_id,))

    spans = cur.fetchall()
    for span in spans:
        attrs = json.loads(span[3]) if span[3] else {}
        print(f"  Span: {span[1]}")
        print(f"    Timestamp: {span[2]}")
        if "gen_ai.usage.input_tokens" in attrs:
            print(f"    Tokens: {attrs['gen_ai.usage.input_tokens']} in / {attrs['gen_ai.usage.output_tokens']} out")
        if "gen_ai.request.model" in attrs:
            print(f"    Model: {attrs['gen_ai.request.model']}")
        print()
else:
    print("No traces found yet — data may take a few minutes to appear.")

Trace detail for: edcd63cb98f5457c8832e532373480d7

  Span: chat openai-gpt-5.4
    Timestamp: 2026-09-24 16:49:51.975402
    Tokens: 783 in / 111 out
    Model: openai-gpt-5.4

  Span: chat openai-gpt-5.4
    Timestamp: 2026-09-24 16:50:03.518337
    Tokens: 1331 in / 245 out
    Model: openai-gpt-5.4

  Span: chat openai-gpt-5.4
    Timestamp: 2026-09-24 16:50:07.913352
    Tokens: 1945 in / 40 out
    Model: openai-gpt-5.4



In [13]:
# Deep dive: reconstruct the full agent conversation chain from a single trace
# This shows every reasoning step — tool calls, tool responses, and final answer.
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
      AND RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT > 2000
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
row = cur.fetchone()

if row:
    tid = row[0]
    cur.execute("""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD_ATTRIBUTES:"gen_ai.input.messages"::STRING AS INPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.output.messages"::STRING AS OUTPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
            RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = %s
        ORDER BY TIMESTAMP
    """, (trace_id,))
    spans = cur.fetchall()
    print(f"Trace {tid} — {len(spans)} gateway call(s)\n")

    for i, span in enumerate(spans):
        msgs_in = json.loads(span[1]) if span[1] else []
        msgs_out = json.loads(span[2]) if span[2] else []
        print(f"═══ Gateway Call {i+1} ({span[3]} in / {span[4]} out tokens) ═══")

        for m in msgs_in:
            role = m.get('role', '?')
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [{role}] {p['content'][:200]}")
                elif 'name' in p:
                    print(f"  [{role}] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
                elif 'response' in p:
                    resp_text = str(p['response'])[:200]
                    print(f"  [tool] ← {resp_text}")

        for m in msgs_out:
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [assistant] {p['content'][:300]}")
                elif 'name' in p:
                    print(f"  [assistant] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
        print()
else:
    print("No multi-turn traces found in last 2 hours.")

Trace aa2602a5839445cb82bc7f10111c80e6 — 4 gateway call(s)

═══ Gateway Call 1 (783 in / 106 out tokens) ═══
  [user] How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? Include actual ROAS and conversion numbers.
  [assistant] → tool_call: query_campaigns({"message": "How did the Q4 Holiday Push campaign perform? Need actual spend, revenue, ROAS, and conversions for the Q4 Holiday Push campaign."})
  [assistant] → tool_call: search_strategy_docs({"columns": ["CONTENT", "TITLE", "ATTRIBUTES"], "limit": 5, "query": "Q4 planning brief targets for Q4 Holiday Push campaign ROAS conversions target g)

═══ Gateway Call 2 (1197 in / 179 out tokens) ═══
  [user] How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? Include actual ROAS and conversion numbers.
  [assistant] → tool_call: query_campaigns({"message": "How did the Q4 Holiday Push campaign perform? Need actual spend, revenue, ROAS, and conversions for the Q4 

## 8. Cost Management

The gateway is a natural unit for cost reporting: every inference request is attributed to the
Snowflake user who made it, with token counts and credit consumption recorded automatically.
This section covers how to monitor what the gateway is spending and how to limit it.

Snowflake offers two spending controls for gateway spend:

| Control | Scope | Enforcement | Use it when |
|---|---|---|---|
| **Shared resource budget** | A group of users identified by a tag, pooled | Notification, plus stored procedures you write | Several teams share the gateway and each needs its own limit |
| **Per-user quota** | Each user individually (limits are never pooled) | Built-in blocking, applied within minutes | You need usage to actually stop, not just alert |

Budget evaluation is periodic (up to 6.5 hours, or one hour with the low-latency option).
Quota blocks apply within minutes.

In [14]:
# Gateway spend by user over the last 7 days
cur.execute("""
    SELECT
        u.NAME AS USER_NAME,
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(g.CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY g
        JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON g.USER_ID = u.USER_ID
        ,LATERAL FLATTEN(input => g.OPERATION_DETAILS) f
    WHERE g.START_TIME > DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY u.NAME, f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Spend by User (last 7 days)\n")
print(f"{'USER':>20} | {'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 90)
for row in rows:
    print(f"{str(row[0]):>20} | {str(row[1]):>20} | {row[2]:>8} | {row[3]:>10} | {row[4]:>10} | {row[5]:>10.6f}")

AI Gateway Spend by User (last 7 days)

                USER |                MODEL | REQUESTS |  IN_TOKENS | OUT_TOKENS |    CREDITS
──────────────────────────────────────────────────────────────────────────────────────────
           PERICKSON |       openai-gpt-5.4 |       97 |      81263 |      10343 |   0.181807


### Shared resource budget

A shared resource budget lets you track gateway spend by team or cost center. The workflow is:

1. **Tag users** with a cost-center or team tag
2. **Create a budget** and add the tag so it tracks only those users
3. **Add the AI Gateway** as a shared resource on the budget
4. **Set a spending limit** — notifications fire when spend approaches it

When a tagged user makes a gateway request, the credits count toward that budget's spending limit.
You can configure notifications and custom actions (stored procedures) that fire at configurable thresholds.

In [15]:
# --- Shared resource budget for gateway spend ---

# 1. Create a tag for cost-center attribution
cur.execute("""CREATE TAG IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER
               COMMENT = 'Team or cost center for gateway budget attribution'""")
print("1. Tag created:", cur.fetchone()[0])

# 2. Tag the current user
cur.execute(f"""ALTER USER "{SNOWFLAKE_USER}" SET TAG CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER = 'ENGINEERING'""")
print("2. User tagged:", cur.fetchone()[0])

# 3. Create the budget
cur.execute("""CREATE SNOWFLAKE.CORE.BUDGET IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET()""")
print("3. Budget created:", cur.fetchone()[0])

# 4. Associate the user tag with the budget
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!SET_USER_TAGS(
    [[(SELECT SYSTEM$REFERENCE('TAG', 'CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER', 'SESSION', 'APPLYBUDGET')),
      'ENGINEERING']],
    'UNION')""")
print("4. User tag set:", cur.fetchone()[0])

# 5. Add AI Gateway as a shared resource
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("5. Shared resource added:", cur.fetchone()[0])

# 6. Set a 500-credit monthly spending limit
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!SET_SPENDING_LIMIT(500)""")
print("6. Spending limit set:", cur.fetchone()[0])

# Verify the budget scope
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!GET_BUDGET_SCOPE()""")
scope = cur.fetchone()[0]
print("\nBudget scope:")
print(json.dumps(json.loads(scope), indent=2))

1. Tag created: COST_CENTER already exists, statement succeeded.
2. User tagged: Statement executed successfully.
3. Budget created: GATEWAY_BUDGET already exists, statement succeeded.
4. User tag set: Successfully removed 1 and set 1 user tag(s) for budget
5. Shared resource added: Successfully operated the shared ai gateway to budget
6. Spending limit set: The spending limit has been updated to 500 credits.

Budget scope:
{
  "resource_tags": {
    "operator": "UNION",
    "tags": []
  },
  "resources": [],
  "shared_resources": [
    {
      "domain": "AI_GATEWAY",
      "id": -1,
      "name": "[ALL-AI GATEWAYS]"
    }
  ],
  "user_tags": {
    "operator": "UNION",
    "tags": [
      {
        "tagDatabase": "CORTEX_GATEWAY_LAB",
        "tagId": 309,
        "tagName": "COST_CENTER",
        "tagSchema": "PUBLIC",
        "tagValues": [
          "ENGINEERING"
        ]
      }
    ]
  }
}


### Per-user quota

A per-user quota enforces hard spending limits at the individual user level. Unlike budgets,
quotas can **block** users automatically when they reach their limit — no custom stored procedure
needed. Blocks are released when the cycle resets (monthly, weekly, or daily).

Per-user quotas support monthly, weekly, and daily limits evaluated independently. A user is
blocked as soon as they hit any one of them. Block enforcement is evaluated within minutes of
a spend event.
> **Warning — this is account-level and affects every user.** The quota below sets a
> 10-credit daily cap and turns on block enforcement for *all* gateway users on the
> account, not just you. On a shared demo or production account that will block real
> people within minutes. Leave `SET_BLOCK_ENFORCEMENT_ENABLED` at `FALSE` unless you
> are on a private account, and run the cleanup cell when you are done.


In [16]:
# --- Per-user quota for gateway spend ---

# 1. Create the quota object
cur.execute("""CREATE SNOWFLAKE.CORE.QUOTA IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA()""")
print("1. Quota created:", cur.fetchone()[0])

# 2. Add AI Gateway as the monitored domain
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("2. Shared resource added:", cur.fetchone()[0])

# 3. Set per-user spending limits at each cycle
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(100)""")           # monthly
print("3a. Monthly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(30, 'WEEKLY')""")  # weekly
print("3b. Weekly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(10, 'DAILY')""")   # daily
print("3c. Daily limit:", cur.fetchone()[0])

# 4. Block enforcement — users are blocked when they hit any limit.
#    Left FALSE by default: this is account-wide and would block every gateway
#    user, not just you. Flip to True only on a private account.
ENABLE_BLOCKING = False
cur.execute(
    f"""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_BLOCK_ENFORCEMENT_ENABLED({ENABLE_BLOCKING})"""
)
print("4. Block enforcement:", cur.fetchone()[0])

# Verify the quota config
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!GET_CONFIG()""")
row = cur.fetchone()
cols = [desc[0] for desc in cur.description]
print("\nQuota config:")
for col, val in zip(cols, row):
    print(f"  {col}: {val}")

1. Quota created: GATEWAY_QUOTA already exists, statement succeeded.
2. Shared resource added: Successfully operated the shared ai gateway to quota
3a. Monthly limit: Per-user limit has been set to 100 credits.
3b. Weekly limit: Per-user weekly limit has been set to 30 credits.
3c. Daily limit: Per-user daily limit has been set to 10 credits.
4. Block enforcement: Block enforcement has been enabled.

Quota config:
  QUOTA_ID: 345
  PER_USER_LIMIT: 100
  REFRESH_TIER: TIER_1H
  ADMIN_EMAILS: None
  ADMIN_LAST_SENT_AT: None
  BLOCK_ENFORCEMENT_ENABLED: True
  PER_USER_LIMIT_DAILY: 10
  PER_USER_BLOCK_NOTIFICATIONS_ENABLED: False
  PER_USER_LIMIT_WEEKLY: 30


## 9. Cleanup

The lab database and warehouse are self-contained, but the budget, quota, user tag, and
gateway spec are **account-level** and outlive a `DROP DATABASE`. Run this to leave the
account as you found it.


In [ ]:
# Account-level teardown. Run this even if you only ran part of the notebook.
cleanup = [
    # Release the per-user quota first so nobody stays blocked
    "CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_BLOCK_ENFORCEMENT_ENABLED(FALSE)",
    "DROP SNOWFLAKE.CORE.QUOTA IF EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA",
    "DROP SNOWFLAKE.CORE.BUDGET IF EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET",
    f'ALTER USER "{SNOWFLAKE_USER}" UNSET TAG CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER',
    "DROP TAG IF EXISTS CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER",
    "DROP DATABASE IF EXISTS CORTEX_GATEWAY_LAB",
    "DROP WAREHOUSE IF EXISTS GATEWAY_LAB_WH",
]

for stmt in cleanup:
    try:
        cur.execute(stmt)
        print("OK  ", stmt[:78])
    except Exception as exc:
        print("skip", stmt[:60], "|", str(exc).split(chr(10))[0][:70])

# The AI Gateway itself is account-level and shared -- it is NOT dropped.
# If you changed its spec, restore the one you captured from SHOW AI GATEWAYS
# before running setup.sql. To turn payload capture back off:
#
# ALTER AI GATEWAY SNOWFLAKE FROM SPECIFICATION $$
# models:
#   - name: '*'
# logging:
#   enabled: true
#   capture_payload:
#     request_response: false
# $$;
print("\nReminder: review SHOW AI GATEWAYS and restore the original spec if needed.")


## Summary

This example demonstrated:

| Component | Role |
|---|---|
| **Cortex AI Gateway** | Centralized LLM routing — one endpoint, configurable routes, built-in observability |
| **LangChain + LangGraph** | Agent framework with ReAct reasoning over MCP tools |
| **MCP (Model Context Protocol)** | Standardized tool interface connecting the agent to Snowflake data services |
| **Cortex Analyst** | Natural language → SQL against the semantic view |
| **Cortex Search** | RAG retrieval over strategy documents |
| **AGENT_TRACE_TABLE** | OpenTelemetry traces for every gateway inference call |
| **AI_GATEWAY_USAGE_HISTORY** | Token usage and credit consumption per model/route |
| **Budgets & Per-user Quotas** | Spending controls — team-level budgets with notifications, or per-user hard blocks |

### Key Takeaways

1. **The gateway is OpenAI-compatible** — any SDK or framework that speaks the OpenAI chat completions API works by pointing `base_url` at `/api/v2/aigateways/snowflake/v1`. Do not confuse this with `/api/v2/cortex/v1`, the standalone Cortex Inference REST API: it answers too, but bypasses the gateway's logging, allowlist, budgets, and quotas.
2. **Model allowlists control access** — the `models` spec restricts which models can be called; use `'*'` for unrestricted or `'claude-*'` to lock down to specific families.
3. **MCP provides a clean tool boundary** — the agent doesn't need Snowflake credentials or SQL knowledge; MCP handles that.
4. **Observability is automatic** — every inference call is traced in `AGENT_TRACE_TABLE('SNOWFLAKE')` with token counts, model metadata, and optional payload capture.
5. **Cost controls are built in** — shared resource budgets track team-level spend with notifications; per-user quotas enforce hard blocks within minutes when a user hits their credit limit.